# Collecter des données OpenAlex

## Découvrir l'API

Jeter un coup d'oeil ici : https://docs.openalex.org/how-to-use-the-api/api-overview

Il faut un token

In [ ]:
import yaml
with open('../creds.yaml', 'r') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

https://openalex.org/works?page=1&filter=title_and_abstract.search:%22computational+social+science%22

Tester sur une requête

In [5]:
import requests

url = "https://api.openalex.org/works"

headers = {
    "Authorization": f"Bearer {config['token-openalex']}"
}

params = {
    "q": "computational social science",
    "per-page": 25,
    "page": 1,
}

r = requests.get(url, headers=headers, params=params)
data = r.json()
#print(data["results"])


Get all the data

In [7]:
import requests
import time


keyword = "computational social science"
per_page = 25     
max_results = None 

url = "https://api.openalex.org/works"

headers = {
    "Authorization": f"Bearer {config['token-openalex']}"
}

cursor = "*"
all_works = []
n = 0

while True:
    params = {
        "filter": f'title_and_abstract.search:"{keyword}"',
        "per-page": per_page,
        "select": "id,type,title,abstract_inverted_index,publication_year,publication_date,open_access,relevance_score",
        "cursor": cursor
    }

    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()
    data = r.json()

    results = data.get("results", [])
    total = data["meta"]["count"]

    if not results:
        break

    all_works.extend(results)
    n += len(results)

    print(f"Fetched {n} / {total} (cursor={cursor})")

    if max_results is not None and n >= max_results:
        break

    cursor = data["meta"].get("next_cursor")
    if not cursor:
        break

    time.sleep(0.2)

print(f"\nDone. Collected {len(all_works)} works total.\n")


Fetched 25 / 2138 (cursor=*)
Fetched 50 / 2138 (cursor=IlsxOTYuNTY5ODksIDE2MDMyMzg0MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMzA5NDQyMzA2OCddIg==)
Fetched 75 / 2138 (cursor=IlsxMjkuMDA0NjEsIDE0MjU5NDU2MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjAwMzc2MTkyMCddIg==)
Fetched 100 / 2138 (cursor=IlsxMDEuMDIxMzgsIDE1NTA0NDgwMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMjkxNjkyMDIzOCddIg==)
Fetched 125 / 2138 (cursor=Ils4My42NjI2OCwgMTY3NDc3NzYwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0MzE4MjQ1MzQ3J10i)
Fetched 150 / 2138 (cursor=Ils2OC4zMTI5NCwgMTY1NDgxOTIwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0MjgyODI0Mjc0J10i)
Fetched 175 / 2138 (cursor=Ils1OC40OTgyMDcsIDE2NzI1MzEyMDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XNDMxNzc3MjI1MiddIg==)
Fetched 200 / 2138 (cursor=Ils1Mi41MzY2MywgMTcwNDA2NzIwMDAwMCwgJ2h0dHBzOi8vb3BlbmFsZXgub3JnL1c0Mzk4MTk5MjExJ10i)
Fetched 225 / 2138 (cursor=Ils0Ni4wODg3NzYsIDEzNDA1ODI0MDAwMDAsICdodHRwczovL29wZW5hbGV4Lm9yZy9XMTExMzY0NTk2J10i)
Fetched 250 / 2138 (cursor=Ils0Mi42NzQ2NywgMTcwNDA2Nz

Sauvegarder les données

In [8]:
import pickle
import os

os.makedirs("data", exist_ok=True)

with open("data/all_works.pkl", "wb") as f:
    pickle.dump(all_works, f)

In [12]:
#all_works[0]

## Mettre sous la forme d'un dataframe

In [15]:
def reconstruct_abstract(inv_index):
    if not inv_index:
        return None
    
    # Determine abstract length
    max_position = max(pos for positions in inv_index.values() for pos in positions)
    abstract_words = [None] * (max_position + 1)

    # Place words in correct positions
    for word, positions in inv_index.items():
        for pos in positions:
            abstract_words[pos] = word

    return " ".join([i for i in abstract_words if i is not None])

#reconstruct_abstract(all_works[200]["abstract_inverted_index"])

In [16]:
import pandas as pd
df = pd.DataFrame(all_works)
df["abstract"] = df["abstract_inverted_index"].apply(reconstruct_abstract)
df.to_csv("data/css_openalex_26022026.csv", index=False)